# 01 · Dense retrieval (bge-m3)

Compare embedding search with the BM25 baseline on the 137 AI Act benchmark questions.

**Runtime:** Runtime → Change runtime type → **T4 GPU**. Run the cells in order.

All the logic lives in `src/regrag/`. This notebook only calls it.

## 1. Setup
Clones the repo on the first run, pulls the latest pushes on later runs, and installs the package.

In [ ]:
REPO_URL = "https://github.com/wu1da/LLM-as-a-Judge-Evaluation-for-an-Agentic-RAG-Assistant-on-EU-Regulatory-Texts.git"
REPO_DIR = "/content/regrag-repo"

import os, sys
if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%pip install -q -e "{REPO_DIR}[dense]"

sys.path.insert(0, f"{REPO_DIR}/src")   # makes `import regrag` work without restarting
DATA_DIR = f"{REPO_DIR}/project-data"

%load_ext autoreload
%autoreload 2

## 2. GPU check
Expected: `True Tesla T4`.

In [ ]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))

## 3. Data + BM25 baseline
Expected: `2149 137`, then `BM25 [0.577, 0.81, 0.905, 0.927]` (same as on the laptop).

In [ ]:
from regrag.parsing import load_articles, load_annexes, make_chunks
from regrag.retrieval import BM25Index, DenseIndex
from regrag.evaluation import load_qa, recall_at_k

chunks = make_chunks(load_articles(DATA_DIR) + load_annexes(DATA_DIR))
qa = load_qa(DATA_DIR)
bm25 = BM25Index(chunks, "en")
print(len(chunks), len(qa))
print("BM25", [round(recall_at_k(bm25, qa, k), 3) for k in (1, 3, 5, 10)])

## 4. Load bge-m3 and build the dense index
The first run downloads the model (about 2 GB). Expected shape: `(1061, 1024)`.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("BAAI/bge-m3", device="cuda")
model.max_seq_length = 512      # our chunks are short; saves memory and time
dense = DenseIndex(chunks, "en", model)
print(dense.embeddings.shape)

## 5. Evaluate
Copy the Dense row into `results.md`.

In [ ]:
print("Dense", [round(recall_at_k(dense, qa, k), 3) for k in (1, 3, 5, 10)])
print([c["chunk_id"] for s, c in dense.search("right to data portability", 3)])